# Collision statistics across stored runs

Aggregate the collision records stored in `Explanations/Samples/*/result.npz`.

The notebook reports two related quantities:

- all entries recorded in `cols`, which counts collision occurrences;
- one terminal outcome per run, using the first entry in `cols` and treating an empty list as `time_out`.

The terminal-outcome distribution is the appropriate one for comparing the overall goal rate with the runs selected by SP-LIME.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%cd ../..
samples_path = Path("Explanations/Samples")

## Load and normalize the result files

In [ ]:
def as_strings(value):
    """Convert a scalar or array-like NPZ value into string entries."""
    values = np.asarray(value, dtype=object).reshape(-1)
    result = []
    for entry in values:
        if isinstance(entry, bytes):
            entry = entry.decode()
        result.append(str(entry))
    return result


def collision_category(entry):
    """Map detailed collision labels to the four analysis categories."""
    if entry == "end":
        return "end"
    if entry.startswith("island_"):
        return "island"
    if entry == "boundary":
        return "boundary"
    return entry


records = []
for result_path in sorted(samples_path.glob("run*/result.npz")):
    with np.load(result_path, allow_pickle=True) as result:
        raw_collisions = as_strings(result["cols"])
        collisions = [collision_category(item) for item in raw_collisions]
        terminal = collisions[0] if collisions else "time_out"
        records.append({
            "run_id": result_path.parent.name,
            "reward": float(np.asarray(result["reward"]).item()),
            "steps": int(np.asarray(result["steps"]).item()),
            "collisions": collisions,
            "terminal": terminal,
        })

runs = pd.DataFrame(records)
print(f"Loaded {len(runs)} runs.")
runs.head()

## Count all recorded collision occurrences

This count uses every entry in every `cols` list. It is useful if a run can record multiple collision events, but it is not a per-run success rate.

In [ ]:
event_counts = (
    runs.explode("collisions")["collisions"]
    .dropna()
    .value_counts()
)

display(event_counts.rename("occurrences").to_frame())
event_counts.plot(kind="bar", title="Recorded collision occurrences", ylabel="count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Per-run terminal outcomes and goal rate

An empty `cols` list is interpreted as `time_out`, matching the collision-category convention used by the run evaluator.

In [ ]:
terminal_counts = runs["terminal"].value_counts()
terminal_table = terminal_counts.rename("runs").to_frame()
terminal_table["fraction"] = terminal_table["runs"] / len(runs)
display(terminal_table)

goal_rate = (runs["terminal"] == "end").mean()
print(f"Goal rate: {goal_rate:.1%} ({(runs['terminal'] == 'end').sum()} / {len(runs)})")

terminal_counts.plot(kind="bar", title="Terminal outcome per run", ylabel="runs")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Optional: compare the SP-LIME-selected runs

Paste the run IDs selected by SP-LIME into `selected_run_ids`. The IDs should be the candidate IDs printed by `sp_lime_expl.ipynb`, for example `run0042`.

In [ ]:
selected_run_ids = []  # e.g. ["run0015", "run0027", "run0042", "run0069"]

if selected_run_ids:
    selected = runs[runs["run_id"].isin(selected_run_ids)].copy()
    missing = sorted(set(selected_run_ids) - set(selected["run_id"]))
    if missing:
        print("Warning: unknown run IDs:", missing)
    display(selected[["run_id", "terminal", "steps", "reward"]])
    selected_counts = selected["terminal"].value_counts()
    display(selected_counts.rename("runs").to_frame())
    if len(selected):
        selected_goal_rate = (selected["terminal"] == "end").mean()
        print(
            f"Selected goal rate: {selected_goal_rate:.1%} "
            f"({(selected['terminal'] == 'end').sum()} / {len(selected)})"
        )
else:
    print("Enter the selected run IDs above to compare the SP-LIME subset.")